In [1]:
!pip -q install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 8.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [92]:
from kaggle_secrets import UserSecretsClient
from google import genai
from google.genai import types
from IPython.display import display, Markdown
import json
import time

api_key = UserSecretsClient().get_secret("Hekayti_Secret")
client = genai.Client(api_key=api_key)

MODEL_NAME = "gemini-3.1-flash-lite"

In [63]:
STORY_PROMPT = """
أنت كاتب قصص أطفال مخصص. اكتب قصة أصلية ومكتملة اعتمادًا على مدخل JSON.
أعد JSON صالحًا فقط، بلا Markdown أو تعليق، وبنفس شكل المخرجات المحدد أدناه.

## شكل المدخل
ستصلك البيانات بهذا الشكل:
{
  "child": {
    "name": "اسم الطفل",
    "age": 7,
    "language": "simplified_fusha",
    "interests": ["اهتمام أو أكثر"],
    "story_goal": "الهدف من القصة",
    "story_type": "نوع القصة",
    "additional_characters": [
      {"name": "اسم الشخصية", "role": "علاقتها بالطفل أو دورها"}
    ]
  },
  "personalization": {
    "dedication": "إهداء أو نص فارغ",
    "preferences": "تفضيلات القصة أو نص فارغ",
    "avoid": ["موضوعات يجب تجنبها"]
  },
  "page_count": 6
}

الحقول المطلوبة: name وage وlanguage وinterests وstory_goal وstory_type وpage_count.
additional_characters وdedication وpreferences وavoid حقول اختيارية؛ قد تكون فارغة.
استخدم فقط المعلومات الموجودة، ولا تخترع معلومات شخصية عن الطفل.

## 1. استخدام بيانات الطفل
- استخدم الاسم والعمر واللغة والاهتمامات والهدف ونوع القصة كما وردت.
- اجعل اهتمام الطفل مؤثرًا في المكان أو الأدلة أو القرارات أو الأدوات أو الحل؛ لا تكتفِ بذكره عرضًا.
- اجعل story_type يحدد طبيعة الأحداث فعلًا، مثل مغامرة خيالية أو قصة صداقة أو اكتشاف وتعلّم.
- ابتكر حبكة أصلية مناسبة لنوع القصة، ولا تكرر قالبًا واحدًا بتغيير الأسماء.
- أظهر صفات الطفل من خلال أفعاله واختياراته، واجعل له قرارًا واحدًا على الأقل يغير مسار القصة.
- إذا وُجدت شخصية إضافية، امنحها دورًا مفيدًا ومتسقًا مع دورها المحدد. يمكنها المساعدة أو تقديم فكرة، لكن لا تجعلها تحل المشكلة الأساسية بدلًا من الطفل.
- اجعل هدف القصة ظاهرًا في سلوك أو قرار خلال الأحداث، لا في موعظة في النهاية.
- تجنب المحاضرات والنجاح الفوري وادعاء أن الطفل تعلم شيئًا دون موقف يثبت ذلك.

## 2. اللغة والسن
استخدم اللغة المحددة بالضبط:
- egyptian_arabic: عامية مصرية طبيعية ومفهومة للطفل.
- simplified_fusha: فصحى مبسطة سليمة، بلا تعبيرات عامية مصرية.
- english: إنجليزية طبيعية مناسبة للطفل.

اضبط المفردات وطول الجمل وتعقيد الأحداث حسب العمر:
- من 3 إلى 4 سنوات: كلمات مألوفة جدًا، جمل قصيرة، وأحداث ملموسة وبسيطة.
- من 5 إلى 6 سنوات: جمل بسيطة، حوار واضح، وسبب ونتيجة مباشران.
- من 7 إلى 8 سنوات: مفردات متنوعة قليلًا، وجمل أطول، ومشكلة تحتاج خطوات تفكير بسيطة.
- من 9 إلى 10 سنوات: مفردات أغنى، وحبكة أكثر تطورًا، ومشاعر ودوافع أوضح، مع الحفاظ على سهولة القراءة.

في كل الأعمار، اجعل اللغة طبيعية واتساقها ثابتًا، وابتعد عن الكلمات الصعبة أو الجمل الطويلة بلا داعٍ.

## ترتيب الحوار
اكتب اسم المتحدث أو فعل القول قبل الكلام المنطوق، مثل:
- العربية: قالت الجدة سلمى: «ابدئي بخطوة صغيرة».
- الإنجليزية: Grandma Salma said, “Start with one small step.”
لا تكتب الحوار أولًا ثم تضيف «قالت الجدة» أو ما شابه بعده.
طبّق هذا الترتيب باتساق في جميع صفحات القصة.

## 3. الحبكة والمنطق
- أنشئ مشكلة واضحة، ومحاولات لحلها، ونتائج أو عقبة، ثم حلًا تدعمه أحداث سابقة.
- اجعل لكل حدث مهم سبب ونتيجة. يجب أن يشارك الطفل في اكتشاف المشكلة أو حلها.
- قبل الحل، أظهر محاولة حقيقية ونتيجة يمكن ملاحظتها، ثم استجابة مناسبة لهذه النتيجة.
- إذا اعتمد الحل على دليل أو معلومة أو قاعدة في عالم القصة، فمهّد لها قبل استخدامها.
- لا تضف دليلًا حاسمًا أو قدرة أو غرضًا في اللحظة الأخيرة لمجرد إنقاذ الحبكة.
- لا تعرض الشك على أنه حقيقة مؤكدة. وضّح كيف عُثر على الدليل ولماذا يدعم الاستنتاج.
- اجعل الحل نتيجة منطقية لقرارات الشخصيات وأفعالها، لا لمصادفة غير مفسرة.

## 4. الواقعية والخيال
- التزم بنوع القصة المحدد في story_type.
- إذا كان النوع خياليًا أو يتضمن السحر، وضّح قواعد العناصر الخيالية مبكرًا والتزم بها حتى النهاية.
- إذا كان النوع واقعيًا، فلا تستخدم أحداثًا خارقة أو سحرية.
- الاهتمام بالحيوانات أو الفضاء أو الديناصورات لا يعني وحده أن القصة خيالية.
- في كل الأنواع، لا تجعل المصادفة أو عنصرًا جديدًا غير ممهد له يحل المشكلة.

## 5. الصفحات واستمرارية المشاهد
- عدد الصفحات المطلوب دقيق ويجب الالتزام به تمامًا.
- الصفحات مشاهد متتالية في قصة واحدة، وليست قصصًا صغيرة منفصلة.
- ابدأ كل صفحة من الحالة التي انتهت بها الصفحة السابقة: مكان الشخصيات، وما يعرفونه، وشعورهم، والأشياء التي معهم.
- يجب أن تضيف كل صفحة حدثًا أو دليلًا أو نتيجة أو تغيرًا في الفهم أو تقدمًا نحو الحل. تجنب التكرار والحشو.
- حافظ على اتساق الشخصيات والأماكن والأغراض والأدلة والمعلومات التي تعرفها الشخصيات.
- لا يظهر شيء أو يختفي أو يتغير دون تفسير.
- اجعل لكل صفحة نص القصة الكامل الخاص بها، لا ملخصًا للمشهد.

## 6. التخصيص والإهداء والتجنب
- أدرج الإهداء والتفضيلات الواردة في المدخل بصورة طبيعية ومناسبة.
- تجنب كل موضوع مذكور في قائمة avoid، ولا تستبدل به موضوعًا مشابهًا مزعجًا.
- اجعل العنوان والملخص والمشاهد ملائمة لنوع القصة وجذابة للقارئ.

## 7. تصميم الشخصيات ووصف الصور
- أضف تصميمًا بصريًا ثابتًا لكل شخصية تظهر في القصة، بما فيها الطفل والشخصيات الإضافية.
- لكل شخصية، اكتب character_id فريدًا، والاسم، والمظهر، والملابس، والسمات المميزة.
- إذا لم يحدد المدخل مظهر الشخصية، ابتكر مظهرًا مناسبًا للقصة والسن، ثم حافظ عليه في جميع الصفحات.
- لا تنسب للطفل صفات جسدية أو شخصية من واقع المدخل لم يذكرها. يمكن اختيار تصميم خيالي مناسب للشخصية داخل القصة.
- في كل صفحة، أضف character_reference_ids التي تطابق معرفات الشخصيات الموجودة فعلًا في ذلك المشهد.
- اكتب image_description واضحًا لفريق الصور يذكر الشخصيات الموجودة، أفعالها، المكان، والأغراض المهمة.
- طابق وصف الصورة مع أحداث نص الصفحة فقط؛ لا تضف فعلًا أو غرضًا أو شخصية غير موجودة في النص.
- حافظ على المظهر والملابس والأغراض المهمة كما وردت في character_designs وبقية الصفحات.
- اجعل الوصف بصريًا ومحددًا، لا تفسيرًا للمشاعر أو ملخصًا للحبكة.

## 8. مراجعة قبل الإخراج
قبل إعادة النتيجة، تحقق من أن:
- بيانات الطفل واهتماماته مدمجة طبيعيًا وتؤثر في أحداث مهمة.
- نوع القصة ظاهر في الأحداث، والهدف يظهر في تصرفات الشخصيات.
- اللغة والسن والعدد الدقيق للصفحات مطابقون للمدخل.
- كل صفحة متصلة بما قبلها، والأحداث والأدلة والحل مترابطة.
- الطفل يشارك في قرار مؤثر وحل المشكلة.
- لا يوجد حل مفاجئ بلا تمهيد أو معلومة مختلقة عن الطفل.
- تصميم كل شخصية واضح وثابت، ومعرفاتها في المشاهد صحيحة.
- وصف الصورة يطابق نص الصفحة.
راجع وعدّل أي جزء لا يحقق ذلك.

## شكل المخرجات
أعد JSON فقط بهذا الشكل، مع استبدال القيم النصية بالمحتوى الفعلي:

{
  "title": "عنوان القصة",
  "summary": "ملخص قصير يوضح الفكرة والمشكلة دون سرد كل الأحداث",
  "character_designs": [
    {
      "character_id": "char_1",
      "name": "اسم الشخصية",
      "appearance": "المظهر والشكل العام",
      "clothing": "الملابس والألوان",
      "distinctive_features": "علامة أو تفصيلة مميزة ثابتة"
    }
  ],
  "pages": [
    {
      "page": 1,
      "scene_title": "عنوان المشهد",
      "text": "نص الصفحة الكامل",
      "image_description": "وصف بصري واضح ودقيق للمشهد",
      "character_reference_ids": ["char_1"]
    }
  ]
}

يجب أن يحتوي pages على العدد المطلوب بالضبط، مع ترقيم متتابع يبدأ من 1.
يجب أن تتضمن character_designs كل شخصية تظهر في القصة، وأن تطابق معرفات الشخصيات في المشاهد.
"""

In [35]:
def make_input(
    name,
    age,
    language,
    interests,
    story_goal,
    story_type,
    page_count,
    additional_characters=None,
    dedication="",
    preferences="",
    avoid=None
):
    return {
        "child": {
            "name": name,
            "age": age,
            "language": language,
            "interests": interests,
            "story_goal": story_goal,
            "story_type": story_type,
            "additional_characters": additional_characters or []
        },
        "personalization": {
            "dedication": dedication,
            "preferences": preferences,
            "avoid": avoid or []
        },
        "page_count": page_count
    }

In [64]:
story_inputs = [
    make_input(
        name="Adam",
        age=4,
        language="english",
        interests=["football", "sports"],
        story_goal="building confidence and feeling less hesitant to join a team activity",
        story_type="a story about friendship",
        page_count=5,
        additional_characters=[
            {"name": "Coach Sam", "role": "his coach who encourages him to take part at his own pace"}
        ],
        dedication="To Adam, a kind and helpful teammate",
        preferences="Natural dialogue and funny moments",
        avoid=["bullying", "deep sadness"]
    ),

    make_input(
    name="آدم",
    age=7,
    language="egyptian_arabic",
    interests=["الحشرات", "التصوير"],
    story_goal="تنمية الصبر والتركيز من خلال ملاحظة التفاصيل الصغيرة",
    story_type="قصة اكتشاف",
    page_count=7,
    additional_characters=[
        {"name": "المعلمة هدى", "role": "معلمة العلوم التي تساعده على طرح الأسئلة وملاحظة التفاصيل"}
    ],
    dedication="إلى آدم، المستكشف الصغير",
    preferences="مغامرة ممتعة ومعلومات بسيطة عن الطبيعة",
    avoid=["التنمر", "المشاهد المخيفة"]
    ),

    make_input(
        name="نور",
        age=10,
        language="simplified_fusha",
        interests=["تصميم الأزياء", "الخياطة", "الرسم"],
        story_goal="تعليم الطفل تقبّل الأخطاء والتعلّم من المحاولة والتعديل",
        story_type="قصة عن تصميم الأزياء والتعلم",
        page_count=10,
        additional_characters=[
            {"name": "الجدة سلمى", "role": "خياطة ماهرة تعلمها استخدام الأدوات بأمان"}
        ],
        dedication="إلى نور، صاحبة الأفكار الجميلة",
        preferences="قصة واقعية ونهاية مشجعة",
        avoid=["التجارب الخطرة", "المشاهد المخيفة"]
    )
]

In [65]:
def validate_story(story, story_input):
    required_top_keys = {
        "title",
        "summary",
        "character_designs",
        "pages"
    }
    missing_keys = required_top_keys - story.keys()

    if missing_keys:
        raise ValueError(f"Missing top-level fields: {sorted(missing_keys)}")

    expected_pages = story_input["page_count"]

    if len(story["pages"]) != expected_pages:
        raise ValueError(
            f"Expected {expected_pages} pages, "
            f"received {len(story['pages'])}."
        )

    expected_numbers = list(range(1, expected_pages + 1))
    actual_numbers = [page.get("page") for page in story["pages"]]

    if actual_numbers != expected_numbers:
        raise ValueError(f"Incorrect page numbering: {actual_numbers}")

    character_ids = set()

    for character in story["character_designs"]:
        for field in (
            "character_id",
            "name",
            "appearance",
            "clothing",
            "distinctive_features"
        ):
            if not character.get(field):
                raise ValueError(
                    f"Character design is missing '{field}': {character}"
                )

        character_ids.add(character["character_id"])

    for page in story["pages"]:
        for field in (
            "scene_title",
            "text",
            "image_description",
            "character_reference_ids"
        ):
            if field not in page:
                raise ValueError(
                    f"Page {page.get('page')} is missing '{field}'."
                )

        unknown_ids = set(page["character_reference_ids"]) - character_ids
        if unknown_ids:
            raise ValueError(
                f"Page {page['page']} uses unknown character IDs: {unknown_ids}"
            )


In [66]:
def generate_story(story_input, temperature=0.4):
    prompt = (
        STORY_PROMPT
        + "\n\nINPUT JSON:\n"
        + json.dumps(story_input, ensure_ascii=False)
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=temperature,
            max_output_tokens=7000
        )
    )

    if not response.text:
        raise ValueError("The model returned an empty response.")

    story = json.loads(response.text)
    validate_story(story, story_input)
    return story

In [93]:
stories = []

for index, story_input in enumerate(story_inputs, start=1):
    print(f"Generating story {index}/{len(story_inputs)}...")

    story = generate_story(story_input)
    stories.append(story)

    print(f"Done: {story['title']} ({len(story['pages'])} pages)")

Generating story 1/3...
Done: Adam's First Goal (5 pages)
Generating story 2/3...
Done: آدم وعدسة الاكتشاف (7 pages)
Generating story 3/3...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [95]:
for index, story in enumerate(stories, start=1):
    display(Markdown(f"# التجربة {index}: {story['title']}"))
    display(Markdown(f"**الملخص:** {story['summary']}"))

    display(Markdown("## تصميم الشخصيات"))

    for character in story["character_designs"]:
         display(Markdown(
            f"**{character['name']} — {character['character_id']}**\n\n"
             f"- المظهر: {character['appearance']}\n"
             f"- الملابس: {character['clothing']}\n"
             f"- علامة مميزة: {character['distinctive_features']}"
         ))

    display(Markdown("## صفحات القصة"))

    for page in story["pages"]:
        references = ", ".join(page["character_reference_ids"]) or "لا توجد"

        display(Markdown(
            f"### صفحة {page['page']}: {page['scene_title']}\n\n"
            f"{page['text']}\n\n"
             f"**وصف الصورة:** {page['image_description']}\n\n"
             f"**مراجع الشخصيات:** {references}"
        ))

# التجربة 1: Adam's First Goal

**الملخص:** Adam, a young boy who loves football, learns to overcome his hesitation and join his friends on the field with the help of Coach Sam.

## تصميم الشخصيات

**Adam — adam_01**

- المظهر: A 4-year-old boy with messy brown hair and bright, curious eyes.
- الملابس: A blue sports jersey, comfortable shorts, and bright red sneakers.
- علامة مميزة: He wears a small wristband with a star pattern.

**Coach Sam — coach_sam_01**

- المظهر: A tall, friendly adult with a kind smile and a baseball cap.
- الملابس: A green whistle around his neck and a sporty tracksuit.
- علامة مميزة: He always carries a colorful football under his arm.

...

### صفحة 3: التركيز والانتظار

أخرج آدم عدسته المكبرة ليرى ما هذا. قالت المعلمة هدى: «حاول أن تثبت يدك وتنتظر، فالتصوير يحتاج إلى هدوء». تنفس آدم بعمق، وثبت كاميرته، وانتظر بصبر حتى توقفت النقطة السوداء عن الحركة.

**وصف الصورة:** آدم يركز عدسة الكاميرا على حشرة صغيرة جداً على ورقة شجر، ملامحه تعبر عن التركيز الشديد.

**مراجع الشخصيات:** adam_01

### صفحة 4: اكتشاف مذهل

التقط آدم الصورة بضغطة خفيفة. نظر إلى شاشة الكاميرا وقال: «يا معلمة، إنها خنفساء صغيرة بنقاط ذهبية! لم أكن لأراها لو لم أنتظر». ابتسمت المعلمة هدى وقالت: «هذا هو سر المستكشف يا آدم، الصبر يكشف لنا جمال التفاصيل».

**وصف الصورة:** آدم ينظر إلى شاشة الكاميرا الصغيرة ويظهر على وجهه تعبير الدهشة، والمعلمة هدى تنظر معه بابتسامة.

**مراجع الشخصيات:** adam_01, huda_02

### صفحة 5: تحدي جديد

اقترح آدم: «هل يمكننا البحث عن شيء آخر؟». قالت المعلمة هدى: «هناك شيء يتحرك تحت أوراق الشجر الجافة، هل يمكنك اكتشافه؟». بدأ آدم يراقب الأرض بهدوء، مستخدماً مهارته الجديدة في التركيز.

**وصف الصورة:** آدم يجلس القرفصاء على العشب ويراقب كومة من أوراق الشجر الجافة، المعلمة هدى تشير بيديها نحو الأرض.

**مراجع الشخصيات:** adam_01, huda_02

### صفحة 6: صورة النملة

رأى آدم نملة تحمل ورقة شجر أكبر منها بكثير. قرر آدم ألا يقترب كثيراً حتى لا يزعجها، بل استخدم خاصية التقريب في كاميرته. قال آدم: «لقد التقطت صورة رائعة لنملة قوية!».

**وصف الصورة:** آدم يمسك الكاميرا من مسافة آمنة، يركز على نملة تحمل قطعة ورقة خضراء صغيرة.

**مراجع الشخصيات:** adam_01

### صفحة 7: المستكشف الصغير

عادت المجموعة إلى الفصل. قالت المعلمة هدى: «لقد تعلمت اليوم أن الصبر هو مفتاح رؤية العالم». نظر آدم إلى صوره الجميلة، وشعر بالفخر لأنه استطاع رؤية ما لا يراه الآخرون بسبب تسرعهم.

**وصف الصورة:** آدم والمعلمة هدى يجلسان في الفصل، وآدم يعرض صوره على شاشة الكاميرا، يبدو عليهما السعادة والرضا.

**مراجع الشخصيات:** adam_01, huda_02

In [30]:
available_models = list(client.models.list())

generate_content_models = []

for model in available_models:
    actions = getattr(model, "supported_actions", None) or []

    if "generateContent" in actions:
        model_name = model.name.replace("models/", "")
        generate_content_models.append(model_name)

print("Models available for generateContent:")
for model_name in generate_content_models:
    print("-", model_name)

Models available for generateContent:
- gemini-2.5-flash
- gemini-2.5-pro
- gemini-2.5-flash-preview-tts
- gemini-2.5-pro-preview-tts
- gemma-4-26b-a4b-it
- gemma-4-31b-it
- gemini-flash-latest
- gemini-flash-lite-latest
- gemini-pro-latest
- gemini-2.5-flash-lite
- gemini-2.5-flash-image
- gemini-3-flash-preview
- gemini-3.1-pro-preview
- gemini-3.1-pro-preview-customtools
- gemini-3.1-flash-lite-preview
- gemini-3.1-flash-lite
- gemini-3-pro-image-preview
- gemini-3-pro-image
- nano-banana-pro-preview
- gemini-3.1-flash-image-preview
- gemini-3.1-flash-image
- gemini-3.1-flash-lite-image
- gemini-nano-banana-2.1
- gemini-3.5-flash
- gemini-3.5-flash-lite
- gemini-omni-flash-preview
- gemini-omni-1.1-flash
- gemini-3.5-transcribe
- gemini-3.6-flash
- gemini-3.7-flash
- gemini-3.8-flash
- gemini-4-argon
- lyria-3-clip-preview
- lyria-3-pro-preview
- lyria-3.5
- gemini-3.1-flash-tts-preview
- gemini-3.8-flash-tts
- gemini-3.8-flash-lite-tts
- gemini-robotics-er-2-preview
- gemini-2.5-co